In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspark-pilot')
    .master('local[*]')
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 18:05:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [3]:
spark.sparkContext.defaultParallelism

15

In [3]:
df_1 = spark.range(4, 200, 2)
df_2 = spark.range(2,200, 4)

In [5]:
df_1.rdd.getNumPartitions()

In [6]:
df_2.rdd.getNumPartitions()

In [4]:
df_3 = df_1.repartition(5)
df_4 = df_2.repartition(7)

In [5]:
df_joined = df_3.join(df_4, on='id')

In [9]:
df_joined.rdd.getNumPartitions()

In [6]:
from pyspark.sql.functions import sum, col

df_sum = df_joined.select(sum(col('id').alias('total_sum')))

In [11]:
df_sum.rdd.getNumPartitions()

In [8]:
df_sum.show()

+--------------------+
|sum(id AS total_sum)|
+--------------------+
|                4998|
+--------------------+



In [7]:
df_sum.explain()

== Physical Plan ==
*(6) HashAggregate(keys=[], functions=[sum(id#0L)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=59]
   +- *(5) HashAggregate(keys=[], functions=[partial_sum(id#0L)])
      +- *(5) Project [id#0L]
         +- *(5) SortMergeJoin [id#0L], [id#1L], Inner
            :- *(2) Sort [id#0L ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(id#0L, 200), ENSURE_REQUIREMENTS, [plan_id=43]
            :     +- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [plan_id=42]
            :        +- *(1) Range (4, 200, step=2, splits=15)
            +- *(4) Sort [id#1L ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(id#1L, 200), ENSURE_REQUIREMENTS, [plan_id=50]
                  +- Exchange RoundRobinPartitioning(7), REPARTITION_BY_NUM, [plan_id=49]
                     +- *(3) Range (2, 200, step=4, splits=15)




In [10]:
df_union = df_sum.union(df_4)

In [11]:
df_union.explain()

== Physical Plan ==
Union
:- *(6) HashAggregate(keys=[], functions=[sum(id#0L)])
:  +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=254]
:     +- *(5) HashAggregate(keys=[], functions=[partial_sum(id#0L)])
:        +- *(5) Project [id#0L]
:           +- *(5) SortMergeJoin [id#0L], [id#1L], Inner
:              :- *(2) Sort [id#0L ASC NULLS FIRST], false, 0
:              :  +- Exchange hashpartitioning(id#0L, 200), ENSURE_REQUIREMENTS, [plan_id=238]
:              :     +- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [plan_id=237]
:              :        +- *(1) Range (4, 200, step=2, splits=15)
:              +- *(4) Sort [id#1L ASC NULLS FIRST], false, 0
:                 +- Exchange hashpartitioning(id#1L, 200), ENSURE_REQUIREMENTS, [plan_id=245]
:                    +- Exchange RoundRobinPartitioning(7), REPARTITION_BY_NUM, [plan_id=244]
:                       +- *(3) Range (2, 200, step=4, splits=15)
+- ReusedExchange [id#18L], Exchange RoundRobinPartitionin

In [15]:
df_union.show()

+--------------------+
|sum(id AS total_sum)|
+--------------------+
|                4998|
|                  14|
|                  46|
|                 130|
|                 138|
|                 162|
|                 198|
|                  18|
|                  50|
|                 190|
|                  26|
|                  42|
|                  54|
|                  66|
|                  98|
|                 106|
|                 186|
|                  10|
|                  38|
|                  62|
+--------------------+
only showing top 20 rows
